# ModelSentry Full Validation
Runs three independent seeds, aggregates the results, and freezes the evidence with SHA-256 hashes. Select a T4 GPU runtime before starting.

In [ ]:
import platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Upload the prepared project archive
Choose `ModelSentry-colab.zip` when prompted.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile
uploaded = files.upload()
archive_name = next(name for name in uploaded if name.lower().endswith('.zip'))
extract_root = Path('/content/modelsentry_upload')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir()
with zipfile.ZipFile(archive_name) as archive:
    archive.extractall(extract_root)
requirements = list(extract_root.rglob('requirements.txt'))
if len(requirements) != 1:
    raise RuntimeError(f'Expected one requirements.txt, found {len(requirements)}')
project_dir = requirements[0].parent
print('Project directory:', project_dir)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project_dir / 'requirements-colab.txt')])
print('Preserved Colab PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## Run the frozen three-seed protocol
This trains on 40,000 images for 8 epochs per seed and runs identical defended and undefended attacks.

In [ ]:
output_dir = Path('/content/validation_results')
if output_dir.exists():
    shutil.rmtree(output_dir)
command = [sys.executable, 'run_validation.py', '--seeds', '42', '7', '123', '--epochs', '8', '--output', str(output_dir)]
subprocess.run(command, cwd=project_dir, check=True)

In [ ]:
import json, pandas as pd
display(pd.read_csv(output_dir / 'per_seed_metrics.csv'))
with (output_dir / 'validation_summary.json').open() as handle:
    summary = json.load(handle)
print(json.dumps(summary, indent=2))
display(summary)

In [ ]:
archive_path = shutil.make_archive('/content/modelsentry_validation', 'zip', output_dir)
print('Created:', archive_path)
files.download(archive_path)